In [21]:
#setup
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mtb_drug_targets/'
!pip install biopython -q
print("Setup complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete!


In [22]:
# Load Top 20 Candidates and Merge Sequences
import pandas as pd
df_top20 = pd.read_csv(BASE + 'results/top20_candidates.csv')
df_proteome = pd.read_csv(BASE + 'results/proteome_final.csv')
df_top20_seq = df_top20.merge(
    df_proteome,
    left_on='query',
    right_on='Protein_ID'
)
print(df_top20_seq.shape)
print(df_top20_seq[['query', 'protein_name']].head())

(20, 16)
                    query                            protein_name
0   sp|I6Y276|Y2993_MYCTU                         Protein Rv2993c
1    sp|P9WKD7|RPIB_MYCTU          Ribose-5-phosphate isomerase B
2  tr|O06178|O06178_MYCTU  Thioesterase domain-containing protein
3  tr|I6X7D4|I6X7D4_MYCTU                       Conserved protein
4   sp|P71592|WHB5A_MYCTU         Transcriptional regulator WhiB5


In [23]:
# Calculate Druggability Properties Function
from Bio.SeqUtils.ProtParam import ProteinAnalysis
def calculate_druggability(sequence):
    analysis = ProteinAnalysis(sequence)
    hydrophobicity = analysis.gravy()
    instability = analysis.instability_index()
    return hydrophobicity, instability
print("Function ready!")

Function ready!


In [24]:
# Calculate Druggability Scores for All 20 Candidates
results = []
for _, row in df_top20_seq.iterrows():
    sequence = row['Protein Sequence']
    hydrophobicity, instability = calculate_druggability(sequence)
    results.append({
        'protein_id': row['query'],
        'protein_name': row['protein_name'],
        'evalue': row['evalue'],
        'hydrophobicity': hydrophobicity,
        'instability': instability
    })
df_scores = pd.DataFrame(results)
print(df_scores)

                protein_id                                       protein_name  \
0    sp|I6Y276|Y2993_MYCTU                                    Protein Rv2993c   
1     sp|P9WKD7|RPIB_MYCTU                     Ribose-5-phosphate isomerase B   
2   tr|O06178|O06178_MYCTU             Thioesterase domain-containing protein   
3   tr|I6X7D4|I6X7D4_MYCTU                                  Conserved protein   
4    sp|P71592|WHB5A_MYCTU                    Transcriptional regulator WhiB5   
5   tr|P71813|P71813_MYCTU                            Uncharacterized protein   
6   tr|P71898|P71898_MYCTU                                  Conserved protein   
7   tr|I6WZ71|I6WZ71_MYCTU  Possible D-3-phosphoglycerate dehydrogenase Se...   
8     sp|P9WPK9|GCS2_MYCTU              Putative glutamate--cysteine ligase 2   
9     sp|P95200|NDHA_MYCTU           Type II NADH:quinone oxidoreductase NdhA   
10    sp|P9WM71|Y090_MYCTU                     Uncharacterized protein Rv0090   
11  tr|Q11064|Q11064_MYCTU  

In [25]:
# Normalize Scores and Calculate Final Ranking
import numpy as np
evalue_norm = df_scores['evalue']
hydro_norm = (df_scores['hydrophobicity'] - df_scores['hydrophobicity'].min()) / \
             (df_scores['hydrophobicity'].max() - df_scores['hydrophobicity'].min())

instab_norm = 1 - (df_scores['instability'] - df_scores['instability'].min()) / \
                   (df_scores['instability'].max() - df_scores['instability'].min())

df_scores['final_score'] = (0.5 * evalue_norm) + (0.3 * hydro_norm) + (0.2 * instab_norm)

df_final = df_scores.sort_values('final_score', ascending=False)

print(df_final[['protein_id', 'protein_name', 'final_score']])

                protein_id                                       protein_name  \
12  tr|P95218|P95218_MYCTU  Probable integral membrane nitrite extrusion p...   
11  tr|Q11064|Q11064_MYCTU                           Probable acyltransferase   
10    sp|P9WM71|Y090_MYCTU                     Uncharacterized protein Rv0090   
7   tr|I6WZ71|I6WZ71_MYCTU  Possible D-3-phosphoglycerate dehydrogenase Se...   
17  tr|P96238|P96238_MYCTU        Possible transcriptional regulatory protein   
9     sp|P95200|NDHA_MYCTU           Type II NADH:quinone oxidoreductase NdhA   
14   sp|O53281|Y3034_MYCTU                 Probable acetyltransferase Rv3034c   
0    sp|I6Y276|Y2993_MYCTU                                    Protein Rv2993c   
16  tr|I6YGT7|I6YGT7_MYCTU                Possible conserved membrane protein   
3   tr|I6X7D4|I6X7D4_MYCTU                                  Conserved protein   
5   tr|P71813|P71813_MYCTU                            Uncharacterized protein   
1     sp|P9WKD7|RPIB_MYCTU  

In [27]:
# Save Final Drug Target Results
df_final.to_csv(BASE + 'results/FINAL_drug_targets.csv', index=False)
print("Final results saved!")
df_top5 = df_final.head(5)
df_top5.to_csv(BASE + 'results/TOP5_drug_targets.csv', index=False)
print("\nTop 5 candidates:")
print(df_top5[['protein_id', 'protein_name', 'final_score']])

Final results saved!

Top 5 candidates:
                protein_id                                       protein_name  \
12  tr|P95218|P95218_MYCTU  Probable integral membrane nitrite extrusion p...   
11  tr|Q11064|Q11064_MYCTU                           Probable acyltransferase   
10    sp|P9WM71|Y090_MYCTU                     Uncharacterized protein Rv0090   
7   tr|I6WZ71|I6WZ71_MYCTU  Possible D-3-phosphoglycerate dehydrogenase Se...   
17  tr|P96238|P96238_MYCTU        Possible transcriptional regulatory protein   

    final_score  
12     0.925600  
11     0.906220  
10     0.845397  
7      0.818531  
17     0.815016  
